In [8]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

#Definir o endereço do site
url="https://books.toscrape.com/"
dados_livros=[]
while url:
    #Fazer o pedido ao site
    resposta= requests.get(url)
    resposta.encoding = "utf-8"

    #Analisar o HTML
    soup = BeautifulSoup(resposta.text, "html.parser")

    #Encontrar todos os livros
    livros=soup.find_all("article", class_="product_pod")
    
    classificacoes={
        "One":1,
        "Two":2,
        "Three":3,
        "Four":4,
        "Five":5
    }
    #Extrair e apresentar os nomes
    for livro in livros:
        nome=livro.find("h3").find("a")["title"]
        #Extrair o preco
        preco=livro.find("p", class_="price_color").text
        #Extrair a disponibilidade em stock
        stock=livro.find("p", class_="instock availability").text.strip()
        #Extrair o rating
        rating=livro.find("p",class_="star-rating")["class"]
        rating = rating[1] # o texto do numero
        rating= classificacoes[rating]
        dados_livros.append({
            "nome":nome,
            "preco":preco,
            "stock":stock,
            "rating":rating
        })
    pagina_seguinte=soup.find("li",class_="next")
    if pagina_seguinte:
        link=pagina_seguinte.find("a")["href"]
        #construir o endereço
        url=urljoin(resposta.url,link)
    else:
        #nao existe mais paginas
        url=None


In [9]:
#criar o dataframe
import pandas as pd
df=pd.DataFrame(dados_livros)


In [10]:
df.head(3)

,nome,preco,stock,rating
0,A Light in the Attic,£51.77,In stock,3
1,Tipping the Velvet,£53.74,In stock,1
2,Soumission,£50.10,In stock,1


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   nome    1000 non-null   str  
 1   preco   1000 non-null   str  
 2   stock   1000 non-null   str  
 3   rating  1000 non-null   int64
dtypes: int64(1), str(3)
memory usage: 31.4 KB


In [ ]:
df["preco"]=df["preco"].str.replace("£","",regex=False)

AttributeError: Can only use .str accessor with string values, not floating

In [13]:
df.head(3)

,nome,preco,stock,rating
0,A Light in the Attic,51.77,In stock,3
1,Tipping the Velvet,53.74,In stock,1
2,Soumission,50.10,In stock,1


In [19]:
df.dtypes

nome          str
preco     float64
stock         str
rating      int64
dtype: object

In [18]:
df["preco"] = df["preco"].astype(float)

In [20]:
df["preco"].mean()

np.float64(35.07035)

In [26]:
df.to_csv("livroswebscraping.csv",sep=";",index=False,encoding="utf-8")